# License and Attribution

**Copyright © 2026 Randy Balzer. All rights reserved.**

This notebook is shared publicly via GitHub for educational, research, and professional portfolio purposes.

**Attribution Requirement**  
If any portion of this work is used, adapted, extended, or incorporated into other projects, research, presentations, or commercial applications, clear and prominent attribution is required in the following form:

> Based on work by Randy Balzer (randy@balzer.io). Original notebook: *agentic_cve_impact_analysis_poc.ipynb*.

For substantial derivative works or any commercial application, please contact randy@balzer.io to discuss appropriate licensing terms.

This material is provided “as is” without warranty of any kind, express or implied. The author disclaims all liability arising from the use or misuse of these notebooks.

# Agentic CVE Impact Analysis PoC

**Multi-Agent Framework for Automated CVE Impact Analysis**

This notebook demonstrates a simple multi-agent pipeline using LangChain that:
1. Ingests a CVE
2. Maps it against a mock enterprise asset inventory
3. Performs mock code-reachability analysis against internal repos
4. Produces a technical + business impact assessment and priority recommendation

Designed as a lightweight proof-of-concept aligned with research on multi-agent vulnerability research and CVE impact analysis.

## Overview

**What this notebook demonstrates**
- A lightweight multi-agent pipeline for automated CVE impact analysis
- Four specialized agents: CVE Ingestion → Asset Mapping → Code Reachability → Impact & Prioritization
- Combines real CVE data (NVD) with enterprise asset context, mock code reachability, and LLM reasoning
- Produces a structured technical + business impact assessment

**Key points**
- Directly addresses the need to automate CVE impact analysis at scale
- Shows practical use of agentic patterns (not just a single LLM call)
- Includes a code-reachability step (important for real prioritization)
- Designed with enterprise reality in mind (asset context, business impact, prioritization)
- Easy to extend (real SCA integration, human approval step, feedback loop)

**Limitations**
- Current version uses small mock inventories and a single LLM
- Real deployments would need CMDB + SCA integration, better reachability analysis, and human review gates
- This is intentionally a focused proof-of-concept, not a production system

## 1. Setup, Imports, and Configuration

In [1]:
!pip install python-dotenv langchain-xai langchain langchain-core -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

import json
import textwrap
import requests
from typing import Dict, Any, List

from IPython.display import display, Markdown, HTML

from langchain_xai import ChatXAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load XAI API Key securely
if not os.environ.get("XAI_API_KEY"):
    os.environ["XAI_API_KEY"] = getpass("Enter your XAI API Key: ")

print("XAI API Key loaded.")

llm = ChatXAI(
    model="grok-4",
    temperature=0.1,
    max_tokens=800
)

XAI API Key loaded.


## 2. Mock Enterprise Asset Inventory

In a real system this would come from a CMDB, asset database, or cloud inventory.

In [3]:
ASSET_INVENTORY = [
    {
        "asset_id": "APP-001",
        "name": "Customer Portal",
        "type": "web_application",
        "tech_stack": ["Apache Struts", "Java 11", "PostgreSQL"],
        "criticality": "High",
        "business_unit": "Retail Banking",
        "internet_facing": True,
        "data_classification": "PII"
    },
    {
        "asset_id": "APP-014",
        "name": "Internal HR Portal",
        "type": "web_application",
        "tech_stack": ["Spring Boot", "Java 17", "MySQL"],
        "criticality": "Medium",
        "business_unit": "Human Resources",
        "internet_facing": False,
        "data_classification": "Internal"
    },
    {
        "asset_id": "SVC-203",
        "name": "Payment Processing Service",
        "type": "microservice",
        "tech_stack": ["Node.js", "Express", "Redis", "OpenSSL"],
        "criticality": "Critical",
        "business_unit": "Payments",
        "internet_facing": True,
        "data_classification": "Financial"
    },
    {
        "asset_id": "INF-088",
        "name": "Legacy File Transfer Server",
        "type": "server",
        "tech_stack": ["OpenSSH 8.2", "Ubuntu 20.04"],
        "criticality": "Medium",
        "business_unit": "Operations",
        "internet_facing": False,
        "data_classification": "Internal"
    },
    {
        "asset_id": "APP-077",
        "name": "AI Document Summarizer",
        "type": "ai_service",
        "tech_stack": ["Python", "LangChain", "OpenAI API", "FastAPI"],
        "criticality": "High",
        "business_unit": "Innovation",
        "internet_facing": True,
        "data_classification": "Confidential"
    }
]

print(f"Loaded {len(ASSET_INVENTORY)} assets")

Loaded 5 assets


## 3. Agent 1 — CVE Ingestion Agent

Fetches basic CVE details from the NVD API (public, no key required for single lookups).

In [ ]:
def fetch_cve(cve_id: str) -> Dict[str, Any]:

    """Fetch CVE details from NVD. Falls back to a mock if the API call fails."""
    url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"

    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        if not data.get("vulnerabilities"):
            raise ValueError("CVE not found")

        cve = data["vulnerabilities"][0]["cve"]

        descriptions = cve.get("descriptions", [])
        description = next((d["value"] for d in descriptions if d["lang"] == "en"), "No description")

        metrics = cve.get("metrics", {})
        cvss_data = {}

        if "cvssMetricV31" in metrics:
            cvss = metrics["cvssMetricV31"][0]["cvssData"]
            cvss_data = {
                "version": "3.1",
                "base_score": cvss.get("baseScore"),
                "base_severity": cvss.get("baseSeverity"),
                "vector": cvss.get("vectorString")
            }
        elif "cvssMetricV2" in metrics:
            cvss = metrics["cvssMetricV2"][0]["cvssData"]
            cvss_data = {
                "version": "2.0",
                "base_score": cvss.get("baseScore"),
                "base_severity": cvss.get("baseSeverity"),
                "vector": cvss.get("vectorString")
            }

        weaknesses = cve.get("weaknesses", [])
        cwes = []

        for w in weaknesses:
            for desc in w.get("description", []):
                if desc.get("lang") == "en":
                    cwes.append(desc.get("value"))

        return {
            "cve_id": cve_id,
            "description": description,
            "cvss": cvss_data,
            "cwes": cwes,
            "published": cve.get("published"),
            "source": "NVD"
        }

    except Exception as e:
        print(f"[!] NVD lookup failed ({e}). Using mock data.")
        return {
            "cve_id": cve_id,
            "description": "Mock: A remote code execution vulnerability in Apache Struts caused by improper input validation in the OGNL expression handling.",
            "cvss": {"version": "3.1", "base_score": 9.8, "base_severity": "CRITICAL", "vector": "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H"},
            "cwes": ["CWE-20"],
            "published": "2023-01-01T00:00:00.000",
            "source": "MOCK"
        }

# Quick test
sample = fetch_cve("CVE-2023-50164")
print(json.dumps(sample, indent=2)[:500] + "...")


{
  "cve_id": "CVE-2023-50164",
  "description": "An attacker can manipulate file upload params to enable paths traversal and under some circumstances this can lead to uploading a malicious file which can be used to perform Remote Code Execution.\nUsers are recommended to upgrade to versions Struts 2.5.33 or Struts 6.3.0.2 or greater to\u00a0fix this issue.",
  "cvss": {
    "version": "3.1",
    "base_score": 9.8,
    "base_severity": "CRITICAL",
    "vector": "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/...


## 4. Agent 2 — Asset Mapping Agent

Uses an LLM to reason about which assets in the inventory are potentially affected by the CVE.

In [ ]:
asset_mapping_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a security analyst specializing in vulnerability-to-asset mapping.
Given a CVE and an enterprise asset inventory, identify which assets are potentially affected.
Be conservative but thorough. Consider technology stack matches, version implications if mentioned, and exposure.
Return a JSON list of affected assets with a short reason for each. If none match, return an empty list.
Only return valid JSON."""),
    ("human", """CVE Information:
{cve_json}

Asset Inventory:
{assets_json}

Return JSON in this format:
[
  {{"asset_id": "...", "name": "...", "reason": "...", "match_confidence": "High|Medium|Low"}}
]""")
])

asset_mapper = asset_mapping_prompt | llm | StrOutputParser()

def map_assets(cve_data: dict) -> list:

    result = asset_mapper.invoke({
        "cve_json": json.dumps(cve_data, indent=2),
        "assets_json": json.dumps(ASSET_INVENTORY, indent=2)
    })

    try:
        cleaned = result.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        return json.loads(cleaned)
        
    except Exception as e:
        print(f"Parse error: {e}")
        print("Raw output:", result)
        return []

## 5. Mock Internal Code & Dependency Inventory

Simulates what an internal SCA / repo search would return. In a real system this would query GitHub Enterprise, Artifactory, Snyk, etc.

In [6]:
CODE_INVENTORY = [
    {
        "repo": "retail/customer-portal",
        "language": "Java",
        "dependencies": [
            {"name": "struts2-core", "version": "2.5.30", "ecosystem": "maven"},
            {"name": "spring-web", "version": "5.3.21", "ecosystem": "maven"},
            {"name": "log4j-core", "version": "2.17.1", "ecosystem": "maven"}
        ],
        "last_scanned": "2026-07-15"
    },
    {
        "repo": "payments/payment-service",
        "language": "JavaScript",
        "dependencies": [
            {"name": "express", "version": "4.18.2", "ecosystem": "npm"},
            {"name": "openssl", "version": "1.1.1", "ecosystem": "system"},
            {"name": "lodash", "version": "4.17.21", "ecosystem": "npm"}
        ],
        "last_scanned": "2026-07-20"
    },
    {
        "repo": "innovation/ai-doc-summarizer",
        "language": "Python",
        "dependencies": [
            {"name": "langchain", "version": "0.1.0", "ecosystem": "pypi"},
            {"name": "openai", "version": "1.3.0", "ecosystem": "pypi"},
            {"name": "fastapi", "version": "0.104.0", "ecosystem": "pypi"}
        ],
        "last_scanned": "2026-07-22"
    },
    {
        "repo": "hr/internal-hr-portal",
        "language": "Java",
        "dependencies": [
            {"name": "spring-boot-starter-web", "version": "3.1.0", "ecosystem": "maven"},
            {"name": "struts2-core", "version": "2.5.26", "ecosystem": "maven"}
        ],
        "last_scanned": "2026-06-01"
    },
    {
        "repo": "ops/legacy-file-transfer",
        "language": "C",
        "dependencies": [
            {"name": "openssh", "version": "8.2p1", "ecosystem": "system"}
        ],
        "last_scanned": "2026-05-10"
    }
]

print(f"Loaded {len(CODE_INVENTORY)} repositories with dependency data")

Loaded 5 repositories with dependency data


## 6. Agent 3 — Code Reachability Agent

Performs a mock "search internal repos" step. Uses the LLM to reason about whether the vulnerable component appears in the code/dependency inventory and how reachable it looks.

In [ ]:
reachability_prompt = ChatPromptTemplate.from_messages([    
    ("system", """You are a security engineer performing code reachability analysis for a CVE.
Given a CVE and an internal code/dependency inventory, identify repositories that appear to contain the vulnerable component or a closely related library.

For each match, assess:
- Whether the vulnerable library/component is present
- Version risk (if version info is available)
- Rough reachability confidence (High / Medium / Low)
- Brief reasoning

Return only valid JSON in this format:
[
  {{
    "repo": "...",
    "matched_component": "...",
    "version_found": "...",
    "reachability": "High|Medium|Low",
    "reason": "..."
  }}
]
If nothing matches, return an empty list []."""),
    ("human", """CVE Information:
{cve_json}

Internal Code & Dependency Inventory:
{code_json}

Return the reachability analysis as JSON.""")
])

reachability_agent = reachability_prompt | llm | StrOutputParser()

def analyze_reachability(cve_data: dict) -> list:

    result = reachability_agent.invoke({
        "cve_json": json.dumps(cve_data, indent=2),
        "code_json": json.dumps(CODE_INVENTORY, indent=2)
    })

    try:
        cleaned = result.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        return json.loads(cleaned)

    except Exception as e:
        print(f"Parse error in reachability agent: {e}")
        print("Raw output:", result)
        return []
        

## 7. Agent 4 — Impact & Prioritization Agent

Produces a structured impact assessment using technical severity, affected assets, **and** code-reachability findings.

In [ ]:
impact_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior vulnerability management analyst.
Given a CVE, the list of potentially affected enterprise assets, and code-reachability findings, produce a concise impact assessment.

Consider:
- Technical severity (CVSS)
- Asset criticality and data classification
- Internet exposure
- Whether the vulnerable component actually appears in internal code (reachability)
- Business unit impact
- Likely exploitability in this environment

Return a clear, structured assessment with:
1. Overall Risk Rating (Critical / High / Medium / Low)
2. Key Affected Assets
3. Code Reachability Summary
4. Business Impact Summary
5. Recommended Priority & Next Actions
6. Short Reasoning"""),
    ("human", """CVE:
{cve_json}

Potentially Affected Assets:
{affected_json}

Code Reachability Findings:
{reachability_json}

Produce the impact assessment.""")
])

impact_analyzer = impact_prompt | llm | StrOutputParser()

def analyze_impact(cve_data: dict, affected_assets: list, reachability: list) -> str:

    return impact_analyzer.invoke({
        "cve_json": json.dumps(cve_data, indent=2),
        "affected_json": json.dumps(affected_assets, indent=2),
        "reachability_json": json.dumps(reachability, indent=2)
    })
    

## 8. Orchestrator — Run the Full Pipeline

**Important:** Run all cells above this one first (Setup → Agents 1–4).

In [ ]:
def _wrap(text: str, width: int = 90) -> str:

    """Wrap long lines for readable notebook output."""
    return textwrap.fill(text, width=width)

def run_cve_impact_pipeline(cve_id: str) -> None:
    
    print("=" * 70)
    print("AGENTIC CVE IMPACT ANALYSIS PIPELINE")
    print(f"Target: {cve_id}")
    print("=" * 70)

    # Agent 1: Ingestion
    print("\n[Agent 1] Ingesting CVE...")
    cve_data = fetch_cve(cve_id)
    print(f"  → Source   : {cve_data['source']}")
    sev = cve_data.get('cvss', {}).get('base_severity', 'N/A')
    score = cve_data.get('cvss', {}).get('base_score', 'N/A')
    print(f"  → Severity : {sev} ({score})")
    print(f"  → Description:")
    print("    " + _wrap(cve_data['description'], 85))

    # Agent 2: Asset Mapping
    print("\n[Agent 2] Mapping to enterprise assets...")
    affected = map_assets(cve_data)
    if affected:
        for a in affected:
            print(f"  → {a.get('asset_id')} | {a.get('name')} | Confidence: {a.get('match_confidence')}")
            print("     " + _wrap(f"Reason: {a.get('reason')}", 80))
    else:
        print("  → No matching assets found in inventory.")

    # Agent 3: Code Reachability
    print("\n[Agent 3] Analyzing code reachability (mock internal repos)...")
    reachability = analyze_reachability(cve_data)
    if reachability:
        for r in reachability:
            print(f"  → Repo: {r.get('repo')}")
            print(f"     Component : {r.get('matched_component')} ({r.get('version_found')})")
            print(f"     Reachability: {r.get('reachability')}")
            print("     " + _wrap(f"Reason: {r.get('reason')}", 80))
    else:
        print("  → No clear code-level matches found in inventory.")

    # Agent 4: Impact Analysis — render as Markdown so it wraps cleanly
    print("\n[Agent 4] Generating impact assessment...")
    assessment = analyze_impact(cve_data, affected, reachability)

    print("\n" + "-" * 70)
    display(Markdown("### Impact Assessment\n\n" + assessment))
    print("-" * 70)
    print("\nPipeline complete.")

# Run against a well-known critical CVE
run_cve_impact_pipeline("CVE-2023-50164")

AGENTIC CVE IMPACT ANALYSIS PIPELINE
Target: CVE-2023-50164

[Agent 1] Ingesting CVE...
  → Source   : NVD
  → Severity : CRITICAL (9.8)
  → Description:
    An attacker can manipulate file upload params to enable paths traversal and under
some circumstances this can lead to uploading a malicious file which can be used to
perform Remote Code Execution. Users are recommended to upgrade to versions Struts
2.5.33 or Struts 6.3.0.2 or greater to fix this issue.

[Agent 2] Mapping to enterprise assets...
  → APP-001 | Customer Portal | Confidence: High
     Reason: Tech stack includes Apache Struts, directly matching the vulnerable
framework (path traversal in file uploads leading to RCE); no version specified
so assumed potentially pre-fix.

[Agent 3] Analyzing code reachability (mock internal repos)...
  → Repo: retail/customer-portal
     Component : struts2-core (2.5.30)
     Reachability: High
     Reason: Direct dependency on vulnerable struts2-core version below 2.5.33
  → Repo: hr/i

### Impact Assessment

1. Overall Risk Rating: **Critical**

2. Key Affected Assets:  
- APP-001 (Customer Portal) – external-facing application  
- Internal HR Portal (hr/internal-hr-portal)

3. Code Reachability Summary:  
High reachability confirmed in both repositories. Direct dependencies on vulnerable `struts2-core` versions (2.5.30 and 2.5.26) exist below the fixed 2.5.33 threshold, with the vulnerable file-upload handling code actively present in the call paths.

4. Business Impact Summary:  
Remote unauthenticated attackers can achieve full RCE on the Customer Portal (publicly reachable) and the Internal HR Portal. This enables complete compromise of customer data, session hijacking, and lateral movement into HR systems containing sensitive employee records.

5. Recommended Priority & Next Actions:  
**Immediate (within 24–48 hours)**:  
- Upgrade both applications to Struts 2.5.33+ (or 6.3.0.2+).  
- Verify no custom file-upload logic bypasses the fix.  
- Deploy WAF rules blocking traversal patterns in upload endpoints as a temporary control.  
- Scan for any other Struts 2 instances in the environment.

6. Short Reasoning:  
CVSS 9.8 RCE with no authentication required, confirmed direct use of exploitable versions, and high code reachability in both an internet-facing and an internal sensitive application.

----------------------------------------------------------------------

Pipeline complete.


## 9. Try Another CVE

Change the CVE ID below to test different scenarios.

In [ ]:
# Examples to try:
# CVE-2021-44228  (Log4Shell)
# CVE-2023-44487  (HTTP/2 Rapid Reset)
# CVE-2024-3094   (XZ Utils backdoor)

# run_cve_impact_pipeline("CVE-2021-44228")

## Notes for Extension / Research

- Replace the mock asset inventory with a real CMDB or cloud inventory API.
- Replace the mock code inventory with real SCA / GitHub / Artifactory queries.
- Add human-in-the-loop approval before final prioritization.
- Track feedback on assessment quality to improve future runs.
- This four-agent pattern directly supports research into multi-agent systems for automated vulnerability research and CVE impact analysis in enterprise environments.